# 02 - Análise de jogadores

Neste notebook usamos os dados já limpos para criar rankings, comparar
posições e clubes, e visualizar relações entre estatísticas.

Tudo aqui usa apenas as colunas que **realmente existem** no dataset:
gols, assistências, cartões e minutos (e suas versões por 90 minutos).

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_players_data
from src.data_cleaning import prepare_players_data, filter_by_minutes
from src.analysis import (
    get_top_scorers,
    get_top_assists,
    get_top_players_by_stat,
    get_average_by_position,
    get_average_by_team,
)

sns.set_style("whitegrid")

df = prepare_players_data(load_players_data("../data/raw/mls_players.csv"))

# Para os rankings, filtramos jogadores com poucos minutos, para não
# comparar diretamente quem jogou pouquíssimo com quem jogou a temporada
# inteira.
df_ranking = filter_by_minutes(df, 500)
print(f"Jogadores com pelo menos 500 minutos: {len(df_ranking)}")

## 1. Ranking de gols (Top 10 artilheiros)

In [ ]:
get_top_scorers(df_ranking, n=10)

## 2. Ranking de assistências (Top 10 assistentes)

In [ ]:
get_top_assists(df_ranking, n=10)

## 3. Ranking de gols e assistências combinados (G+A)

Este dataset não possui xG nem xAG (o CSV é a tabela "Standard Stats" do
FBref) -- por isso os rankings ficam limitados às estatísticas
disponíveis: gols, assistências e a soma G+A.

In [ ]:
get_top_players_by_stat(df_ranking, "g_a", n=10)

## 4. Rankings por 90 minutos

Permitem comparar jogadores com volumes de minutos diferentes de forma mais justa.

In [ ]:
get_top_players_by_stat(df_ranking, "gls_per90", n=10)

In [ ]:
get_top_players_by_stat(df_ranking, "ast_per90", n=10)

## 5. Média de gols por 90 minutos, por posição

In [ ]:
media_posicao = get_average_by_position(df_ranking, "gls_per90")
media_posicao

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
media_posicao.plot(kind="bar", ax=ax, color="#1f6feb")
ax.set_title("Média de gols por 90 minutos, por posição")
ax.set_ylabel("Gols / 90 min (média)")
plt.show()

## 6. Média de gols por clube (Top 10 clubes)

In [ ]:
media_clube = get_average_by_team(df_ranking, "gls_per90")
media_clube.head(10)

## 7. Top 10 artilheiros (gráfico de barras)

In [ ]:
top10_artilheiros = get_top_scorers(df_ranking, n=10)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=top10_artilheiros, x="gls", y="player", color="#1f6feb", ax=ax)
ax.set_title("Top 10 artilheiros")
ax.set_xlabel("Gols")
ax.set_ylabel("")
plt.show()

## 8. Top 10 assistentes (gráfico de barras)

In [ ]:
top10_assistentes = get_top_assists(df_ranking, n=10)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=top10_assistentes, x="ast", y="player", color="#1f6feb", ax=ax)
ax.set_title("Top 10 assistentes")
ax.set_xlabel("Assistências")
ax.set_ylabel("")
plt.show()

## 9. Distribuição de gols

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df_ranking["gls"], bins=15, color="#1f6feb", ax=ax)
ax.set_title("Distribuição de gols (jogadores com 500+ minutos)")
ax.set_xlabel("Gols")
plt.show()

## 10. Distribuição de idade

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df_ranking["age"].dropna(), bins=15, color="#1f6feb", ax=ax)
ax.set_title("Distribuição de idade (jogadores com 500+ minutos)")
ax.set_xlabel("Idade")
plt.show()

## 11. Gols x xG

O dataset **não possui a coluna xG** (Expected Goals). Em vez de simular
ou inventar essa informação, mostramos abaixo como o projeto reage a essa
ausência: de forma explícita, sem quebrar.

In [ ]:
from src.data_cleaning import check_required_columns

resultado = check_required_columns(df_ranking, ["gls", "xg"])
print(resultado)

if "xg" not in df_ranking.columns:
    print("\nNão é possível calcular o gráfico Gols x xG porque a coluna 'xG' não está disponível neste dataset.")

## 12. Chutes x gols

Da mesma forma, este dataset não possui uma coluna de chutes ('Sh').

In [ ]:
resultado = check_required_columns(df_ranking, ["gls", "sh"])
if "sh" not in df_ranking.columns:
    print("Não é possível calcular o gráfico Chutes x Gols porque a coluna 'Sh' não está disponível neste dataset.")

## 13. Heatmap de correlação entre as estatísticas disponíveis

In [ ]:
colunas_numericas = [
    "age", "min", "gls", "ast", "g_a", "crdy", "crdr",
    "gls_per90", "ast_per90", "g_a_per90",
]
colunas_numericas = [c for c in colunas_numericas if c in df_ranking.columns]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df_ranking[colunas_numericas].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlação entre estatísticas")
plt.show()

## 14. Estatísticas médias por posição

In [ ]:
colunas_stats = ["gls_per90", "ast_per90", "g_a_per90", "crdy", "crdr"]
colunas_stats = [c for c in colunas_stats if c in df_ranking.columns]

df_ranking.groupby("pos")[colunas_stats].mean().round(3)

## Conclusão

- Os rankings e comparações funcionam bem com as estatísticas de ataque
  (gols, assistências) disponíveis no CSV.
- Estatísticas avançadas (xG, xAG, chutes, passes progressivos) **não
  existem neste arquivo** -- o código identifica isso e avisa, em vez de
  inventar valores.
- A correlação mostra, como esperado, forte relação entre `gls` e
  `gls_per90`, e entre `ast` e `ast_per90` (são a mesma informação em
  escalas diferentes).
- O próximo notebook (`03_clustering.ipynb`) usa essas estatísticas por 90
  minutos para agrupar jogadores com estilos semelhantes.